# 04 - Migración de Scraper a API de Mercadona

Este notebook explora y documenta la migración del scraping basado en Selenium a las APIs REST de Mercadona.

## Objetivos
- Explorar las APIs documentadas de Mercadona
- Extraer productos de todas las categorías
- Comparar datos obtenidos vs scraping actual
- Generar el mismo formato CSV para compatibilidad con la web app

In [2]:
import requests
import pandas as pd
import json
from pathlib import Path
from time import sleep
from typing import Dict, List, Any

## 1. Configuración de APIs

In [3]:
# Configuración base
BASE_URL = "https://tienda.mercadona.es/api"
PARAMS = {
    "lang": "es",
    "wh": "mad1"  # Almacén Madrid 1 - cambiar según necesidad
}

# Headers para simular navegador
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
    "Accept": "application/json",
}

## 2. Exploración de Endpoints

In [4]:
# Probar endpoint de HOME (no categories)
# Este endpoint sí contiene la estructura real de categorías
response = requests.get(f"{BASE_URL}/home/", params=PARAMS, headers=HEADERS)
print(f"Status: {response.status_code}")
print(f"\nEstructura de respuesta:")
home_data = response.json()
print(json.dumps(home_data, indent=2, ensure_ascii=False)[:2000])  # Primeros 2000 chars

Status: 200

Estructura de respuesta:
{
  "sections": [
    {
      "layout": "notification",
      "content": {
        "title": "Identifícate y añade tu dirección para conocer la próxima entrega disponible",
        "type": "info",
        "event_key": "availability_no_address"
      }
    },
    {
      "layout": "banner",
      "content": {
        "title": "Productos del momento",
        "subtitle": "Selección de productos destacados",
        "items": [
          {
            "id": 130,
            "title": "Caprichos de Pascua",
            "campaign_id": "easter-treats",
            "image_url": "https://prod-mercadona.imgix.net/images/cf9ec2f110d2a9c523d75deddfe80969.jpg?h=500",
            "text_color": "rgba(255, 255, 255, 1.0)",
            "bg_colors": [
              "rgba(0, 0, 0, 0.6)",
              "rgba(255, 255, 255, 0.0)"
            ],
            "button_color": "rgba(153, 213, 62, 1.0)",
            "api_path": "/home/sections/97b40ad2-14a8-4236-a7e4-ca1f447f4

In [5]:
# Explorar una categoría específica
# Ejemplo: categoría 112
category_id = 112
response = requests.get(f"{BASE_URL}/categories/{category_id}/", params=PARAMS, headers=HEADERS)
print(f"Status: {response.status_code}")
category_detail = response.json()
print(json.dumps(category_detail, indent=2, ensure_ascii=False)[:2000])

Status: 200
{
  "id": 112,
  "name": "Aceite, vinagre y sal",
  "order": 7,
  "layout": 1,
  "published": true,
  "is_extended": false,
  "categories": [
    {
      "id": 420,
      "name": "Aceite de oliva",
      "order": 7,
      "layout": 2,
      "published": true,
      "is_extended": false,
      "image": null,
      "subtitle": null,
      "products": [
        {
          "id": "4241",
          "slug": "aceite-oliva-04o-hacendado-garrafa",
          "limit": 999,
          "badges": {
            "is_water": false,
            "requires_age_check": false
          },
          "status": null,
          "packaging": "Garrafa",
          "published": true,
          "share_url": "https://tienda.mercadona.es/product/4241/aceite-oliva-04o-hacendado-garrafa",
          "thumbnail": "https://prod-mercadona.imgix.net/images/3b8cde7b3cb069ee0316029012cf8562.jpg?fit=crop&h=300&w=300",
          "categories": [
            {
              "id": 12,
              "name": "Aceite, espec

## 3. Funciones de Extracción

In [6]:
def get_home_data() -> Dict[str, Any]:
    """Obtiene datos de la página home que incluye todas las categorías."""
    response = requests.get(f"{BASE_URL}/home/", params=PARAMS, headers=HEADERS)
    response.raise_for_status()
    return response.json()

def get_category_detail(category_id: int) -> Dict[str, Any]:
    """Obtiene detalle de una categoría específica."""
    response = requests.get(f"{BASE_URL}/categories/{category_id}/", params=PARAMS, headers=HEADERS)
    response.raise_for_status()
    return response.json()

def extract_category_ids_from_home(home_data: Dict[str, Any]) -> set:
    """
    Extrae todos los IDs de categorías desde el endpoint /home/.
    Navega por sections -> categories para encontrar los IDs reales.
    """
    category_ids = set()
    
    # Navegar por las secciones
    sections = home_data.get('sections', [])
    for section in sections:
        # Cada sección puede tener categorías
        categories = section.get('categories', [])
        for category in categories:
            if 'id' in category:
                category_ids.add(category['id'])
            
            # Las categorías pueden tener subcategorías anidadas
            if 'categories' in category:
                for subcat in category['categories']:
                    if 'id' in subcat:
                        category_ids.add(subcat['id'])
    
    return category_ids

def extract_all_category_ids_recursive(category_data: Dict[str, Any], collected_ids: set = None) -> set:
    """
    Extrae recursivamente todos los IDs de categorías y subcategorías.
    """
    if collected_ids is None:
        collected_ids = set()

    # Añadir el ID actual si existe
    if 'id' in category_data:
        collected_ids.add(category_data['id'])

    # Buscar subcategorías recursivamente
    if 'categories' in category_data:
        for subcategory in category_data['categories']:
            extract_all_category_ids_recursive(subcategory, collected_ids)

    return collected_ids

def extract_products_from_category(category_data: Dict[str, Any], parent_category: str = "") -> List[Dict[str, Any]]:
    """
    Extrae productos de una categoría, navegando recursivamente por subcategorías.
    """
    products = []

    # Si esta categoría tiene productos directamente
    if 'products' in category_data:
        for product in category_data['products']:
            product['parent_category'] = parent_category or category_data.get('name', '')
            products.append(product)

    # Navegar recursivamente por subcategorías
    if 'categories' in category_data:
        current_name = parent_category or category_data.get('name', '')
        for subcategory in category_data['categories']:
            products.extend(extract_products_from_category(subcategory, current_name))

    return products

## 4. Extracción Completa de Datos

In [7]:
# DESCUBRIMIENTO: Los IDs válidos están distribuidos en el rango 1-1500
# Vamos a escanear todo el rango para encontrar todas las categorías válidas

print("Escaneando IDs de categorías desde 1 hasta 1500...")
print("Esto puede tardar unos minutos...\n")

valid_category_ids = []

# Escanear en bloques de 100 para mostrar progreso
for block_start in range(1, 1501, 100):
    block_end = min(block_start + 100, 1501)
    print(f"Bloque {block_start:4d}-{block_end:4d}...", end=" ", flush=True)
    
    found_in_block = 0
    
    for cat_id in range(block_start, block_end):
        try:
            response = requests.get(
                f"{BASE_URL}/categories/{cat_id}/", 
                params=PARAMS, 
                headers=HEADERS,
                timeout=5
            )
            if response.status_code == 200:
                valid_category_ids.append(cat_id)
                found_in_block += 1
            
            # Rate limiting más agresivo para evitar bloqueos
            sleep(0.05)
            
        except Exception as e:
            # Ignorar errores de conexión
            pass
    
    print(f"{found_in_block:2d} encontradas")

print(f"\n✓ Total categorías válidas encontradas: {len(valid_category_ids)}")
print(f"\nPrimeros 30 IDs: {sorted(valid_category_ids)[:30]}")
print(f"Últimos 10 IDs: {sorted(valid_category_ids)[-10:]}")

Escaneando IDs de categorías desde 1 hasta 1500...
Esto puede tardar unos minutos...

Bloque    1- 101... 53 encontradas
Bloque  101- 201... 65 encontradas
Bloque  201- 301... 31 encontradas
Bloque  301- 401...  0 encontradas
Bloque  401- 501...  0 encontradas
Bloque  501- 601...  0 encontradas
Bloque  601- 701...  0 encontradas
Bloque  701- 801...  0 encontradas
Bloque  801- 901...  2 encontradas
Bloque  901-1001...  0 encontradas
Bloque 1001-1101...  0 encontradas
Bloque 1101-1201...  0 encontradas
Bloque 1201-1301...  0 encontradas
Bloque 1301-1401...  0 encontradas
Bloque 1401-1501...  0 encontradas

✓ Total categorías válidas encontradas: 151

Primeros 30 IDs: [27, 28, 29, 31, 32, 34, 36, 37, 38, 40, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 56, 58, 59, 60, 62, 64, 65]
Últimos 10 IDs: [234, 235, 237, 238, 239, 241, 243, 244, 884, 897]


In [8]:
# Ahora extraer productos de todas las categorías válidas
print("Extrayendo productos de todas las categorías...\n")
all_products = []

for i, cat_id in enumerate(sorted(valid_category_ids), 1):
    try:
        category_data = get_category_detail(cat_id)
        
        # Extraer productos recursivamente
        products = extract_products_from_category(category_data)
        
        if products:
            all_products.extend(products)
            cat_name = category_data.get('name', f'ID {cat_id}')
            print(f"[{i}/{len(valid_category_ids)}] {cat_name}: {len(products)} productos")
        
        sleep(0.2)  # Rate limiting
        
    except Exception as e:
        print(f"[{i}/{len(valid_category_ids)}] ID {cat_id}: Error - {str(e)[:50]}")

print(f"\n✓ Total productos extraídos: {len(all_products)}")

# Deduplicar por ID
unique_products = {}
for product in all_products:
    if 'id' in product:
        unique_products[product['id']] = product

print(f"✓ Productos únicos: {len(unique_products)}")

Extrayendo productos de todas las categorías...

[1/151] Fruta: 47 productos
[2/151] Lechuga y ensalada preparada: 30 productos
[3/151] Verdura: 96 productos
[4/151] Pescado fresco: 37 productos
[5/151] Marisco: 35 productos
[6/151] Pescado congelado: 51 productos
[7/151] Salazones y ahumados: 19 productos
[8/151] Cerdo: 37 productos
[9/151] Aves y pollo: 51 productos
[10/151] Vacuno: 12 productos
[11/151] Conejo y cordero: 5 productos
[12/151] Embutido: 11 productos
[13/151] Hamburguesas y picadas: 26 productos
[14/151] Empanados y elaborados: 33 productos
[15/151] Arreglos: 14 productos
[16/151] Carne congelada: 12 productos
[17/151] Aves y jamón cocido: 35 productos
[18/151] Chopped y mortadela: 10 productos
[19/151] Jamón serrano: 26 productos
[20/151] Embutido curado: 50 productos
[21/151] Bacón y salchichas: 24 productos
[22/151] Queso untable, fresco y especialidades: 42 productos
[23/151] Queso curado, semicurado y tierno: 44 productos
[24/151] Queso lonchas, rallado y en porci

## 5. Análisis de Estructura de Datos

In [9]:
# Analizar estructura de un producto de ejemplo
# Usar unique_products en lugar de all_products para evitar duplicados
all_products = list(unique_products.values())

if all_products:
    print("Ejemplo de producto:")
    print(json.dumps(all_products[0], indent=2, ensure_ascii=False))

Ejemplo de producto:
{
  "id": "3819",
  "slug": "platano-canarias-igp-pieza",
  "limit": 999,
  "badges": {
    "is_water": false,
    "requires_age_check": false
  },
  "status": null,
  "packaging": "Pieza",
  "published": true,
  "share_url": "https://tienda.mercadona.es/product/3819/platano-canarias-igp-pieza",
  "thumbnail": "https://prod-mercadona.imgix.net/images/e4a37940916985bf5ca166e266580c37.jpg?fit=crop&h=300&w=300",
  "categories": [
    {
      "id": 1,
      "name": "Fruta y verdura",
      "level": 0,
      "order": 306
    }
  ],
  "display_name": "Plátano de Canarias IGP",
  "unavailable_from": null,
  "price_instructions": {
    "iva": null,
    "is_new": false,
    "is_pack": false,
    "pack_size": null,
    "unit_name": null,
    "unit_size": 0.15,
    "bulk_price": "2.30",
    "unit_price": "0.35",
    "approx_size": true,
    "size_format": "kg",
    "total_units": null,
    "unit_selector": true,
    "bunch_selector": false,
    "drained_weight": null,
    "se

In [10]:
# Convertir a DataFrame
df = pd.DataFrame(all_products)
print(f"Shape: {df.shape}")
print(f"\nColumnas disponibles ({len(df.columns)}):")
for col in sorted(df.columns.tolist()):
    print(f"  - {col}")

print(f"\n{'='*60}")
print("ANÁLISIS DETALLADO DE CAMPOS")
print(f"{'='*60}\n")

# Mostrar tipos de datos y ejemplos
for col in df.columns:
    print(f"\n{col}:")
    print(f"  Tipo: {df[col].dtype}")
    print(f"  Nulls: {df[col].isna().sum()}/{len(df)}")
    
    # Mostrar ejemplo del primer valor no nulo
    first_value = df[col].dropna().iloc[0] if not df[col].dropna().empty else None
    if isinstance(first_value, dict):
        print(f"  Ejemplo (dict keys): {list(first_value.keys())}")
    elif isinstance(first_value, list):
        print(f"  Ejemplo (list): {len(first_value)} elementos")
        if first_value:
            print(f"    Primer elemento: {first_value[0]}")
    else:
        print(f"  Ejemplo: {str(first_value)[:100]}")

df.head(3)

Shape: (4300, 16)

Columnas disponibles (16):
  - badges
  - categories
  - display_name
  - id
  - limit
  - packaging
  - parent_category
  - price_instructions
  - published
  - share_url
  - slug
  - status
  - thumbnail
  - unavailable_from
  - unavailable_weekdays
  - video_url

ANÁLISIS DETALLADO DE CAMPOS


id:
  Tipo: object
  Nulls: 0/4300
  Ejemplo: 3819

slug:
  Tipo: object
  Nulls: 0/4300
  Ejemplo: platano-canarias-igp-pieza

limit:
  Tipo: int64
  Nulls: 0/4300
  Ejemplo: 999

badges:
  Tipo: object
  Nulls: 0/4300
  Ejemplo (dict keys): ['is_water', 'requires_age_check']

status:
  Tipo: object
  Nulls: 4300/4300
  Ejemplo: None

packaging:
  Tipo: object
  Nulls: 554/4300
  Ejemplo: Pieza

published:
  Tipo: bool
  Nulls: 0/4300
  Ejemplo: True

share_url:
  Tipo: object
  Nulls: 0/4300
  Ejemplo: https://tienda.mercadona.es/product/3819/platano-canarias-igp-pieza

thumbnail:
  Tipo: object
  Nulls: 0/4300
  Ejemplo: https://prod-mercadona.imgix.net/images/e4a37940916

,id,slug,limit,badges,status,packaging,published,share_url,thumbnail,categories,display_name,unavailable_from,price_instructions,unavailable_weekdays,parent_category,video_url
0,3819,platano-canarias-igp-pieza,999,"{'is_water': False, 'requires_age_check': False}",None,Pieza,True,https://tienda.mercadona.es/product/3819/plata...,https://prod-mercadona.imgix.net/images/e4a379...,"[{'id': 1, 'name': 'Fruta y verdura', 'level':...",Plátano de Canarias IGP,None,"{'iva': None, 'is_new': False, 'is_pack': Fals...",[],Fruta,NaN
1,3824,banana-pieza,999,"{'is_water': False, 'requires_age_check': False}",None,Pieza,True,https://tienda.mercadona.es/product/3824/banan...,https://prod-mercadona.imgix.net/images/69edef...,"[{'id': 1, 'name': 'Fruta y verdura', 'level':...",Banana,None,"{'iva': None, 'is_new': False, 'is_pack': Fals...",[],Fruta,NaN
2,3132,platano-macho-pieza,999,"{'is_water': False, 'requires_age_check': False}",None,Pieza,True,https://tienda.mercadona.es/product/3132/plata...,https://prod-mercadona.imgix.net/images/c22baa...,"[{'id': 1, 'name': 'Fruta y verdura', 'level':...",Plátano macho,None,"{'iva': 4, 'is_new': False, 'is_pack': False, ...",[],Fruta,NaN


In [15]:
# Mostrar todo en df[df["id"] == "21557"].thumbnail

df[df["id"] == "21557"]

,id,slug,limit,badges,status,packaging,published,share_url,thumbnail,categories,display_name,unavailable_from,price_instructions,unavailable_weekdays,parent_category,video_url
1682,21557,bebida-lactea-desnatada-natural-l-casei-0-mg-0...,999,"{'is_water': False, 'requires_age_check': False}",None,None,True,https://tienda.mercadona.es/product/21557/bebi...,https://prod-mercadona.imgix.net/images/7bad9e...,"[{'id': 11, 'name': 'Postres y yogures', 'leve...",Bebida láctea desnatada natural L-casei 0% MG ...,None,"{'iva': None, 'is_new': False, 'is_pack': True...",[],Yogures líquidos,NaN


In [21]:
# Mostrar todo en df[df["id"] == "21557"].thumbnail

df[df["id"] == "21557"].thumbnail.iloc[0]

'https://prod-mercadona.imgix.net/images/7bad9ecb51987296b00c6ceba72c41e7.jpg?fit=crop&h=300&w=300'

## 6. Transformación a Formato Compatible

In [25]:
# TRANSFORMACIÓN COMPLETA - Extraer TODOS los campos útiles

def extract_price_info(price_instructions):
    """Extrae información de precio del dict price_instructions."""
    if not isinstance(price_instructions, dict):
        return pd.Series({
            'unit_price': None,
            'bulk_price': None,
            'reference_price': None,
            'unit_size': None,
            'size_format': None,
            'reference_format': None,
            'previous_unit_price': None,
            'price_decreased': None,
            'is_new': None,
            'is_pack': None,
            'pack_size': None,
            'tax_percentage': None
        })
    
    return pd.Series({
        'unit_price': price_instructions.get('unit_price'),
        'bulk_price': price_instructions.get('bulk_price'),
        'reference_price': price_instructions.get('reference_price'),
        'unit_size': price_instructions.get('unit_size'),
        'size_format': price_instructions.get('size_format'),
        'reference_format': price_instructions.get('reference_format'),
        'previous_unit_price': price_instructions.get('previous_unit_price', '').strip() if price_instructions.get('previous_unit_price') else None,
        'price_decreased': price_instructions.get('price_decreased'),
        'is_new': price_instructions.get('is_new'),
        'is_pack': price_instructions.get('is_pack'),
        'pack_size': price_instructions.get('pack_size'),
        'tax_percentage': price_instructions.get('tax_percentage')
    })

def extract_badges(badges):
    """Extrae badges del producto."""
    if not isinstance(badges, dict):
        return pd.Series({'is_water': None, 'requires_age_check': None})
    
    return pd.Series({
        'is_water': badges.get('is_water'),
        'requires_age_check': badges.get('requires_age_check')
    })

def extract_categories(categories):
    """Extrae información de categorías."""
    if not isinstance(categories, list) or not categories:
        return pd.Series({'category_id': None, 'category_name': None, 'category_level': None})
    
    # Tomar la primera categoría (o la de menor nivel)
    cat = categories[0]
    return pd.Series({
        'category_id': cat.get('id'),
        'category_name': cat.get('name'),
        'category_level': cat.get('level')
    })

# Aplicar transformaciones
print("Transformando datos...")

# Extraer campos básicos
df_transformed = pd.DataFrame({
    'id': df['id'],
    'slug': df['slug'],
    'display_name': df['display_name'],
    'packaging': df['packaging'],
    'thumbnail': df['thumbnail'],
    'share_url': df['share_url'],
    'published': df['published'],
    'status': df['status'],
    'limit': df['limit'],
    'parent_category': df.get('parent_category', '')
})

# Extraer información de precio
price_info = df['price_instructions'].apply(extract_price_info)
df_transformed = pd.concat([df_transformed, price_info], axis=1)

# Extraer badges
badges_info = df['badges'].apply(extract_badges)
df_transformed = pd.concat([df_transformed, badges_info], axis=1)

# Extraer categorías
categories_info = df['categories'].apply(extract_categories)
df_transformed = pd.concat([df_transformed, categories_info], axis=1)

# Extraer unavailable_weekdays (convertir lista a string)
df_transformed['unavailable_weekdays'] = df['unavailable_weekdays'].apply(
    lambda x: ','.join(map(str, x)) if isinstance(x, list) and x else ''
)

# Unavailable_from
df_transformed['unavailable_from'] = df['unavailable_from']

print(f"\n✓ Transformación completa")
print(f"Columnas en df_transformed: {len(df_transformed.columns)}")
print(f"Filas: {len(df_transformed)}")

print("\nColumnas finales:")
for col in df_transformed.columns:
    print(f"  - {col}")

df_transformed.head()

Transformando datos...

✓ Transformación completa
Columnas en df_transformed: 29
Filas: 4300

Columnas finales:
  - id
  - slug
  - display_name
  - packaging
  - thumbnail
  - share_url
  - published
  - status
  - limit
  - parent_category
  - unit_price
  - bulk_price
  - reference_price
  - unit_size
  - size_format
  - reference_format
  - previous_unit_price
  - price_decreased
  - is_new
  - is_pack
  - pack_size
  - tax_percentage
  - is_water
  - requires_age_check
  - category_id
  - category_name
  - category_level
  - unavailable_weekdays
  - unavailable_from


,id,slug,display_name,packaging,thumbnail,share_url,published,status,limit,parent_category,...,is_pack,pack_size,tax_percentage,is_water,requires_age_check,category_id,category_name,category_level,unavailable_weekdays,unavailable_from
0,3819,platano-canarias-igp-pieza,Plátano de Canarias IGP,Pieza,https://prod-mercadona.imgix.net/images/e4a379...,https://tienda.mercadona.es/product/3819/plata...,True,None,999,Fruta,...,False,NaN,4.000,False,False,1,Fruta y verdura,0,,None
1,3824,banana-pieza,Banana,Pieza,https://prod-mercadona.imgix.net/images/69edef...,https://tienda.mercadona.es/product/3824/banan...,True,None,999,Fruta,...,False,NaN,4.000,False,False,1,Fruta y verdura,0,,None
2,3132,platano-macho-pieza,Plátano macho,Pieza,https://prod-mercadona.imgix.net/images/c22baa...,https://tienda.mercadona.es/product/3132/plata...,True,None,999,Fruta,...,False,NaN,4.000,False,False,1,Fruta y verdura,0,,None
3,3313,uva-blanca-sin-semillas-bandeja,Uva blanca sin semillas,Bandeja,https://prod-mercadona.imgix.net/images/8c2869...,https://tienda.mercadona.es/product/3313/uva-b...,True,None,999,Fruta,...,False,NaN,4.000,False,False,1,Fruta y verdura,0,,None
4,3321,uva-roja-sin-semillas-bandeja,Uva roja sin semillas,Bandeja,https://prod-mercadona.imgix.net/images/2ea068...,https://tienda.mercadona.es/product/3321/uva-r...,True,None,999,Fruta,...,False,NaN,4.000,False,False,1,Fruta y verdura,0,,None


In [29]:
df_transformed.columns

Index(['id', 'slug', 'display_name', 'packaging', 'thumbnail', 'share_url',
       'published', 'status', 'limit', 'parent_category', 'unit_price',
       'bulk_price', 'reference_price', 'unit_size', 'size_format',
       'reference_format', 'previous_unit_price', 'price_decreased', 'is_new',
       'is_pack', 'pack_size', 'tax_percentage', 'is_water',
       'requires_age_check', 'category_id', 'category_name', 'category_level',
       'unavailable_weekdays', 'unavailable_from'],
      dtype='object')

In [33]:
df_transformed.is_new.value_counts()

is_new
False    4300
Name: count, dtype: int64

## 7. Comparación con Datos Actuales

In [35]:
# Cargar datos actuales del scraper
current_data_path = Path('../data/processed/products_macro.csv')

if current_data_path.exists():
    df_current = pd.read_csv(current_data_path)
    print(f"Datos actuales (scraper): {df_current.shape}")
    print(f"Datos nuevos (API): {df_transformed.shape}")
    print(f"\nColumnas actuales:")
    print(df_current.columns.tolist())
    print(f"\nColumnas nuevas:")
    print(df_transformed.columns.tolist())
else:
    print("No se encontró archivo de datos actual")

Datos actuales (scraper): (4793, 10)
Datos nuevos (API): (4300, 29)

Columnas actuales:
['id', 'Category', 'name', 'subtitle', 'price', 'discount_price', 'main_image_url', 'secondary_image_url', 'nutritional_info', 'novedad']

Columnas nuevas:
['id', 'slug', 'display_name', 'packaging', 'thumbnail', 'share_url', 'published', 'status', 'limit', 'parent_category', 'unit_price', 'bulk_price', 'reference_price', 'unit_size', 'size_format', 'reference_format', 'previous_unit_price', 'price_decreased', 'is_new', 'is_pack', 'pack_size', 'tax_percentage', 'is_water', 'requires_age_check', 'category_id', 'category_name', 'category_level', 'unavailable_weekdays', 'unavailable_from']
